# 01 - Data Exploration
Beach Bird Detection Dataset (YOLO format, 1 class: bird).
Checks split sizes, visualises annotations, and computes dataset statistics
(birds per image, box sizes) to guide method design.

## Setup and split sizes

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

DATA = Path("../dataset")

def load_labels(img_path, split):
    """Return array of [class, cx, cy, w, h] (normalised) for an image."""
    lbl = DATA / "labels" / split / (img_path.stem + ".txt")
    if not lbl.exists() or lbl.stat().st_size == 0:
        return np.zeros((0, 5))          # negative image (no birds)
    return np.loadtxt(lbl, ndmin=2)

for split in ["train", "val", "test"]:
    n_img = len(list((DATA / "images" / split).iterdir()))
    n_lbl = len(list((DATA / "labels" / split).iterdir()))
    print(f"{split}: {n_img} images, {n_lbl} label files")

## Visualise ground-truth boxes

In [ ]:
def show(img_path, split):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    boxes = load_labels(img_path, split)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img)
    for _, cx, cy, w, h in boxes:
        x1, y1 = (cx - w/2) * W, (cy - h/2) * H
        ax.add_patch(patches.Rectangle((x1, y1), w*W, h*H,
                                       fill=False, edgecolor="red", lw=1))
    ax.set_title(f"{img_path.name} — {len(boxes)} birds")
    ax.axis("off")
    plt.show()

imgs = sorted((DATA / "images" / "train").iterdir())
show(imgs[0], "train")

## Dataset statistics

In [ ]:
counts, widths, heights = [], [], []
for img_path in imgs:
    b = load_labels(img_path, "train")
    counts.append(len(b))
    if len(b):
        H, W = cv2.imread(str(img_path)).shape[:2]
        widths += list(b[:, 3] * W)
        heights += list(b[:, 4] * H)

counts = np.array(counts)
print(f"Images with no birds: {(counts == 0).sum()}")
print(f"Birds per image: mean {counts.mean():.1f}, max {counts.max()}")
print(f"Box size (px): median {np.median(widths):.0f} x {np.median(heights):.0f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].hist(counts, bins=30); axs[0].set_title("Birds per image")
axs[1].hist(widths, bins=50); axs[1].set_title("Box width (px)")
plt.show()

## Image resolution

In [ ]:
sizes = [cv2.imread(str(p)).shape[:2] for p in imgs[:50]]
print(set(sizes))

## Statistics per split

In [ ]:
def split_stats(split):
    paths = sorted((DATA / "images" / split).iterdir())
    counts, widths = [], []
    for p in paths:
        b = load_labels(p, split)
        counts.append(len(b))
        if len(b):
            H, W = cv2.imread(str(p)).shape[:2]
            widths += list(b[:, 3] * W)
    counts = np.array(counts)
    print(f"{split:5s}: {len(paths)} images | no birds: {(counts == 0).sum()} "
          f"| birds/img mean {counts.mean():.1f}, max {counts.max()} "
          f"| median box width {np.median(widths):.0f}px")

for s in ["train", "val", "test"]:
    split_stats(s)